# Visualize Rates
### Import libraries

In [ ]:
import os
import sys
import pickle
import importlib
import numpy as np
import pandas as pd
import tensorflow as tf

from tqdm.notebook import tqdm

Visualization libraries

In [ ]:
import matplotlib as mpl
from matplotlib import pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
# %matplotlib qt

Global paths

In [ ]:
# WORKSPACE
WORKSPACE = os.environ['WORKSPACE_PATH']
# PyCOMET
PyCOMET = WORKSPACE + '/ML/PyCOMET/pycomet/'
if importlib.util.find_spec('pycomet') is None:
    sys.path.append(os.path.abspath(PyCOMET))
# SurQCT
SurQCT = WORKSPACE + '/ML/PyCOMET/pycomet/examples/SurQCT/'
sys.path.append(os.path.abspath(SurQCT))
# Current Working Directory
CWD = os.getcwd()

In [ ]:
# Output directory
out_dir = CWD + '/Output/VisualizeRates/'
if not os.path.exists(out_dir):
    os.makedirs(out_dir)

### Initialize PyCOMET model
Working case

In [ ]:
case_dir = WORKSPACE + '/ML/PyCOMET/run/SurQCT/mimmo/O3_UMN/KInel/case.01/'
sys.path.append(os.path.abspath(case_dir))

In [ ]:
os.chdir(case_dir)

Import additional libraries

In [ ]:
from pycomet.utilities import utils
from pycomet.apps import functions as fun
# Inputs
import inputs
from inputs.general import flags

Compile model

In [ ]:
# Load network architecture
net = fun._net(
    input_mod=inputs
)
# Finalize model
for k in ('train', 'save'):
    flags[k] = 0
flags['load'] = 1
model = fun._model(
    flags=flags,
    input_mod=inputs,
    net=net
)

In [ ]:
os.chdir(CWD)

### Initialize SurQCT parameters
SurQCT model inputs

In [ ]:
surqct_inp = case_dir + '/data_gen/InputData.py'
sys.path.append(os.path.dirname(surqct_inp))

import InputData

In [ ]:
# Modifiy test input parameters
InputData.MinRateValue = 1.e-20
InputData.TTranVecTest = [1500., 5000., 10000., 15000.]

### Generating data
Import `RatesType` module

In [ ]:
RatesType = __import__(
    f'surqct.src.RatesType.{InputData.NbAtoms}Atoms.{InputData.RatesType}',
    globals(),
    locals(),
    [InputData.RatesType],
    0
)

Read levels data

In [ ]:
LevelsData, DiatData, NLevels = RatesType.read_levelsinfo(InputData, InputData.TTranVecTest)

Sampling levels

In [ ]:
# Predefined
iLevels = [1, 10, 32, 34, 35, 81, 82, 98, 1000, 2000, 3000, 4581]

# Random
iLevels = iLevels + list(np.random.choice(NLevels[0], size=10, replace=False) + 1)

# By rotational quantum number
JqnVec = [0]
for jqn in JqnVec:
    iLevels = iLevels + [idx+1 for idx in np.arange(NLevels[0]) if (DiatData[0]['jqn'].to_numpy(int)[idx] == jqn)]

InputData.iLevelsVecTest = np.unique(iLevels)

Generate testing data

In [ ]:
x_true, y_true = RatesType.generate_testingdata(InputData)
data_true = pd.concat([x_true, y_true], axis=1)

Generate prediction data

In [ ]:
x_pred, y_pred = RatesType.generate_predictiondata(InputData, model.net, Labeled=True)
data_pred = pd.concat([x_pred, y_pred], axis=1)

### Plotting rates

In [ ]:
for T in InputData.TTranVecTest:
    paths = {k: out_dir + f'/T{int(T)}K/{k}/' for k in ('2D','3D')}
    for path in paths.values():
        if not os.path.exists(path):
            os.makedirs(path)
    for iIdx in tqdm(InputData.iLevelsVecTest, desc='Temperature T = ' + str(int(T)) + 'K'):
        # True values
        TrueData = data_true.loc[(data_true['TTran_i'] == T) & (data_true['Idx_i'] == iIdx-1) & (data_true['KExcit'] >= InputData.MinRateValue)]
        TrueData = (TrueData['Idx_j'].to_numpy().astype(int), TrueData['KExcit'].to_numpy())
        # Pred values
        PredData = data_pred.loc[(data_pred['TTran_i'] == T) & (data_pred['Idx_i'] == iIdx-1) & (data_pred['KExcit'] >= InputData.MinRateValue)]
        PredData = (PredData['Idx_j'].to_numpy().astype(int), PredData['KExcit'].to_numpy())
        # Plotting
        RatesType.plot_prediction3D(paths['3D'], iIdx, DiatData, TrueData, PredData)
        plt.close()
        RatesType.plot_prediction2D(paths['2D'], iIdx, DiatData, TrueData, PredData)
        plt.close()